# PHYS 690 - Lecture 18: MCMC Sampling with Metropolis-Hastings and emcee

## Sampling the Ramsey posterior without a grid

Lecture 17 evaluated the binomial posterior for the reduced parameter vector $\boldsymbol{\psi}=(\tau,C)$ on a two-dimensional grid. That grid-cell approximation was useful for seeing the posterior directly, but a grid with 100 values for each of ten parameters would require $100^{10}$ evaluations.

Markov chain Monte Carlo (MCMC) instead generates correlated draws from the same posterior density. In this lecture we use the same fixed data, binomial likelihood, uniform prior bounds, and fixed calibration values for $\omega$ and $\phi$ as Lecture 17. The target does not change; the method for exploring it changes. We implement random-walk Metropolis-Hastings, inspect its trace, and reproduce the same target with `emcee`. Lecture 19 develops formal convergence diagnostics. The dashed generating values in the figures are known only because this is synthetic data.

## A Bayesian joke before we sample

[xkcd 1132, *Frequentists vs. Bayesians*](https://xkcd.com/1132/) jokes about how prior knowledge can change the interpretation of a surprising observation. In this lecture, the prior is explicit and quantitative: MCMC combines it with the likelihood to explore the posterior. The comic is a prompt for discussion, not a substitute for checking the model and the data.

[![xkcd 1132: Frequentists vs. Bayesians](https://imgs.xkcd.com/comics/frequentists_vs_bayesians.png)](https://xkcd.com/1132/)


## Learning goals

By the end of class, you should be able to:

1. derive and implement the Metropolis-Hastings acceptance rule;
2. explain how retained chain states represent a posterior distribution;
3. summarize marginal and joint posterior uncertainty;
4. reproduce the inference with the production sampler `emcee`;
5. compare posterior summaries from different portions of an `emcee` run; and
6. explain how sampling several parameters reveals correlations and joint uncertainty.

## Environment setup: install `emcee` and select the notebook kernel

This lecture now uses `emcee`, which is pinned in the repository's `requirements.txt`. Install packages into this repository's `.venv` so the notebook kernel can import the same versions. Run these commands from the repository root in the VS Code terminal. If `.venv` already exists, skip the `venv` creation command and activate it.

**macOS / Linux:**

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

**Windows using Git Bash:**

```bash
python -m venv .venv
source .venv/Scripts/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

In VS Code, use **Select Kernel** at the top right of the notebook and choose **PHYS 690 (.venv)**. After installing or updating packages, choose **Restart Kernel** and then run the verification cell below. The Python path it prints should point inside this repository's `.venv`, and `emcee` should report version 3.1.6. If you use JupyterLab, select the same kernel from its kernel menu and restart it there. See the [course VS Code environment guide](../resources/vscode.md) for more details.

In [ ]:
# Confirm this notebook is using the repository environment with emcee installed.
import sys
import emcee

print("Notebook Python:", sys.executable)
print("emcee version:", emcee.__version__)

In [ ]:
# NumPy handles numerical calculations, pandas organizes observations and samples,
# Matplotlib makes figures, and SciPy supplies the binomial probability function.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import binom

# Use consistent figure defaults throughout the notebook.
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

# Evaluate the Ramsey excited-state probability at one or many delay times.
def ramsey_probability(t, contrast, tau, omega, phase):
    t = np.asarray(t)
    probability = 0.5 * (
        1.0 + contrast * np.exp(-t / tau) * np.cos(omega * t + phase)
    )
    return np.clip(probability, 1.0e-12, 1.0 - 1.0e-12)


# These are the parameter values used only to generate the synthetic observations.
true_parameters = {
    "contrast": 0.78,
    "tau": 2.40,
    "omega": 2.0 * np.pi * 0.85,
    "phase": 0.25,
}

# Generate the same reproducible finite-shot data in Lectures 17-19.
data_rng = np.random.default_rng(69017)
delays = np.linspace(0.0, 5.0, 31)
shots = np.full(delays.size, 250, dtype=int)
generating_probability = ramsey_probability(delays, **true_parameters)
counts = data_rng.binomial(shots, generating_probability)

# Keep times, counts, shot totals, and probabilities together in one data table.
ramsey_data_df = pd.DataFrame({
    "time_us": delays,
    "shots": shots,
    "excited_counts": counts,
    "measured_probability": counts / shots,
    "generating_probability": generating_probability,
})

omega_fixed = true_parameters["omega"]
phase_fixed = true_parameters["phase"]

# Use the same uniform prior support as Lecture 17.
tau_prior_min = 0.4  # microseconds
tau_prior_max = 6.0  # microseconds
contrast_prior_min = 0.0
contrast_prior_max = 1.0

In [ ]:
# Return the log probability of the observed counts for proposed tau and contrast.
def binomial_log_likelihood(tau, contrast, data_df=ramsey_data_df):
    # Return -infinity outside the physical parameter region.
    if tau <= 0.0 or contrast < 0.0 or contrast > 1.0:
        return -np.inf

    probability = ramsey_probability(
        data_df["time_us"].to_numpy(), contrast, tau, omega_fixed, phase_fixed
    )
    # Read the observed number of trials and excited-state counts from the table.
    n = data_df["shots"].to_numpy()
    k = data_df["excited_counts"].to_numpy()

    # Evaluate the standard binomial log-PMF for each observed count.
    return np.sum(binom.logpmf(k, n, probability))


# Evaluate the constant uniform prior inside the allowed parameter ranges.
def log_prior(tau, contrast):
    # Use independent uniform priors over a physically allowed teaching range.
    if (
        tau_prior_min <= tau <= tau_prior_max
        and contrast_prior_min <= contrast <= contrast_prior_max
    ):
        return -np.log((tau_prior_max - tau_prior_min)
                        * (contrast_prior_max - contrast_prior_min))
    return -np.inf


# Combine the log prior and log likelihood; the normalization is unnecessary for MCMC.
def log_posterior(parameters):
    tau, contrast = parameters
    return log_prior(tau, contrast) + binomial_log_likelihood(tau, contrast)

## Reminder: the Ramsey measurement model

At each delay $t$, the experiment records how many of $n$ two-level systems are found in the excited state. The measured fraction estimates the excited-state probability, modeled here as

$$
p_e(t\mid C,\tau,\omega,\phi)=\frac{1}{2}\left[1+C e^{-t/\tau}\cos(\omega t+\phi)\right].
$$

Here $t$ is the delay in microseconds, $\omega$ is the angular frequency in radians per microsecond, and $\phi$ is a phase offset in radians. The oscillation comes from Ramsey phase accumulation, while the exponential envelope describes loss of coherence. At each delay, $n=250$ independent shots produce an excited-state count $k$; the measured fraction is $k/n$. In the synthetic data, $C=0.78$ and $\tau=2.40\,\mu\mathrm{s}$ are the parameters we infer; the calibration values $\omega=2\pi(0.85\,\mu\mathrm{s}^{-1})$ and $\phi=0.25$ are held fixed. Each count is drawn from a binomial distribution with $n=250$ trials. The dashed curve below is the known generating probability, available because these data are simulated.

In [ ]:
# Draw a smooth model curve over the delay range covered by the measurements.
plot_times = np.linspace(
    ramsey_data_df["time_us"].min(),
    ramsey_data_df["time_us"].max(),
    500,
)
plot_probability = ramsey_probability(plot_times, **true_parameters)

# Approximate binomial standard errors for the observed fractions.
measured_fraction = ramsey_data_df["measured_probability"]
trial_count = ramsey_data_df["shots"]
fraction_se = np.sqrt(measured_fraction * (1.0 - measured_fraction) / trial_count)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.errorbar(
    ramsey_data_df["time_us"],
    measured_fraction,
    yerr=fraction_se,
    fmt="o",
    capsize=2,
    label="simulated measurements",
)
ax.plot(plot_times, plot_probability, "-", lw=2, label="generating probability")
ax.set(
    xlabel="Ramsey delay $t$ ($\u03bcs$)",
    ylabel="Excited-state probability",
    ylim=(0.0, 1.0),
    title="Ramsey data and the generating model",
)
ax.legend()
plt.show()

## Part 1: Posterior summaries as Monte Carlo averages

Lecture 17 defined the posterior $p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})$ for the Ramsey parameters $\boldsymbol{\psi}=(\tau,C)$ given the observed data $D_{\mathrm{obs}}$. A posterior summary can be written as an average over that distribution. For any quantity $g$ calculated from the parameters—for example, $g(\boldsymbol{\psi})=\tau$—

$$
\text{posterior average of }g=\int g(\boldsymbol{\psi})p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})\,d\boldsymbol{\psi}.
$$

If $\boldsymbol{\psi}^{(s)}$ denotes the parameter pair at step $s$ and $S$ is the number of retained steps, the average can be estimated by

$$
\frac{1}{S}\sum_{s=1}^{S}g(\boldsymbol{\psi}^{(s)}).
$$

This connects to Lecture 15: both methods estimate an average from samples. The difference is that MCMC samples are generated sequentially. The next state depends on the current state, so nearby samples are often similar. We call this sequence a **Markov chain**. This dependence means that the raw number of retained steps is not the same as the number of independent draws. Lecture 19 will quantify how this affects numerical precision.

## Part 2: Random-walk Metropolis-Hastings

We first work through the accept/reject decision using a simple one-dimensional posterior, then apply the same steps to the two-parameter Ramsey posterior. A **proposal** is a candidate parameter value generated from the current state. For this random walk, we add Gaussian noise:

$$
\boldsymbol{\psi}'=\boldsymbol{\psi}+\boldsymbol{\eta},\qquad
\boldsymbol{\eta}\sim\mathcal{N}(\boldsymbol{0},\Sigma_q).
$$

Here $\Sigma_q$ is the proposal covariance matrix: it sets the typical step size in $\tau$ and $C$ (and any correlation between the proposed steps). Our code uses independent steps, so $\Sigma_q$ is diagonal and its diagonal entries are the squared proposal scales.

For a general proposal density $q(\boldsymbol{\psi}'\mid\boldsymbol{\psi})$, the Metropolis-Hastings rule accounts for how likely it is to propose each direction:

$$
\alpha=\min\left(1,
\frac{p(\boldsymbol{\psi}'\mid D_{\mathrm{obs}})q(\boldsymbol{\psi}\mid\boldsymbol{\psi}')}
{p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})q(\boldsymbol{\psi}'\mid\boldsymbol{\psi})}
\right).
$$

Our Gaussian random walk is symmetric: proposing $\boldsymbol{\psi}'$ from $\boldsymbol{\psi}$ has the same density as proposing $\boldsymbol{\psi}$ from $\boldsymbol{\psi}'$. The two $q$ terms therefore cancel, leaving the ratio of the Lecture 17 posterior values. If the proposed posterior is higher, accept the move; if it is lower, accept with probability equal to the ratio. Define the log-posterior change

$$
\Delta\log p=\log p(\boldsymbol{\psi}'\mid D_{\mathrm{obs}})-\log p(\boldsymbol{\psi}\mid D_{\mathrm{obs}}).
$$

Draw $U$ uniformly between zero and one and accept when $U<\alpha$ (equivalently, $\log U<\min(0,\Delta\log p)$). If the proposal is rejected, record the current state again. In Lecture 17, $p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})\propto p(D_{\mathrm{obs}}\mid\boldsymbol{\psi})p(\boldsymbol{\psi})$: within the uniform prior bounds, the prior is constant; outside them, the posterior is zero and the proposal is rejected. The unknown overall normalization cancels in the ratio.

For the hand-worked calculation, $x$ is a single toy parameter and $D$ denotes its data. We stipulate the simple posterior shape $p(x\mid D)\propto e^{-x^2/2}$ so we can focus on the accept/reject arithmetic. The candidates and uniform draws are supplied.

In [ ]:
# The toy log posterior is known up to an additive normalization constant.
def toy_log_posterior(x):
    return -0.5 * x**2

# This completed function carries out the accept/reject decision for one proposal.
def accept_or_reject(current, proposal, uniform_draw):
    current_log_posterior = toy_log_posterior(current)
    proposal_log_posterior = toy_log_posterior(proposal)
    log_posterior_change = proposal_log_posterior - current_log_posterior
    log_acceptance_limit = min(0.0, log_posterior_change)
    accepted = np.log(uniform_draw) < log_acceptance_limit
    next_state = proposal if accepted else current
    return next_state, accepted, log_posterior_change, log_acceptance_limit

# These are illustrative candidates from a symmetric random-walk proposal.
# The first move is accepted; the second is rejected.
worked_moves = [(1.0, 0.40), (3.0, 0.10)]  # (candidate, uniform draw)
# Apply the same accept/reject calculation twice, carrying the accepted state forward.
current_state = 0.0
worked_rows = []
for candidate, uniform_draw in worked_moves:
    old_state = current_state
    current_state, accepted, log_change, log_limit = accept_or_reject(
        old_state, candidate, uniform_draw
    )
    worked_rows.append({
        "current": old_state,
        "candidate": candidate,
        "delta_log_posterior": log_change,
        "uniform_draw": uniform_draw,
        "acceptance_probability": np.exp(log_limit),
        "log(U)": np.log(uniform_draw),
        "accepted": accepted,
        "next_state": current_state,
    })

# Store the arithmetic for both proposals in a table for easy inspection.
worked_example_df = pd.DataFrame(worked_rows)
worked_example_df

### See the decision on the posterior

The curve below shows the one-dimensional posterior shape $p(x\mid D)\propto e^{-x^2/2}$. The upper panels mark the current point and proposed point. Their height ratio gives $\alpha=\min[1,p(x\prime\mid D)/p(x\mid D)]$. The lower number lines compare that threshold with the sampled $U$: accept when $U<\alpha$. For the Ramsey sampler, use the Lecture 17 posterior explicitly: $\alpha=\min[1,p(\boldsymbol{\psi}\prime\mid D_{\mathrm{obs}})/p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})]$.

In [ ]:
# Plot the one-dimensional posterior shape and the accept/reject threshold for each move.
toy_x = np.linspace(-3.5, 3.5, 500)
toy_curve_df = pd.DataFrame({
    "x": toy_x,
    "posterior_shape": np.exp(-0.5 * toy_x**2),
})

fig, axes = plt.subplots(
    2, 2, figsize=(9, 5),
    gridspec_kw={"height_ratios": [3.0, 1.0]},
    constrained_layout=True,
)

for column, move in enumerate(worked_example_df.itertuples(index=False)):
    density_ax = axes[0, column]
    decision_ax = axes[1, column]

    # Show the posterior shape and mark its values at the current/proposed states.
    density_ax.plot(toy_curve_df["x"], toy_curve_df["posterior_shape"], color="C0")
    current_density = np.exp(toy_log_posterior(move.current))
    proposal_density = np.exp(toy_log_posterior(move.candidate))
    density_ax.vlines(move.current, 0.0, current_density, color="C1", ls="--")
    density_ax.vlines(move.candidate, 0.0, proposal_density, color="C2", ls="--")
    density_ax.scatter([move.current], [current_density], color="C1", zorder=3,
                       label="current state")
    density_ax.scatter([move.candidate], [proposal_density], color="C2", marker="D",
                       zorder=3, label="proposed state")
    decision = "accepted" if move.accepted else "rejected"
    density_ax.set_title(
        f"Move {column + 1}: $\\alpha$={move.acceptance_probability:.3f}, "
        f"$U$={move.uniform_draw:.2f} → {decision}"
    )
    density_ax.set_ylabel("Posterior shape $p(x|D)$")
    density_ax.set_xlim(-3.5, 3.5)
    density_ax.legend(loc="upper right", fontsize=8)

    # Shade values of U that would accept; the vertical line is the sampled U.
    decision_ax.axvspan(0.0, move.acceptance_probability, color="C2", alpha=0.25)
    decision_ax.axvline(move.uniform_draw, color="C3", lw=2, label="$U$ drawn")
    decision_ax.set_xlim(0.0, 1.0)
    decision_ax.set_ylim(0.0, 1.0)
    decision_ax.set_yticks([])
    decision_ax.set_xlabel("Uniform draw $U$; shaded values are accepted")
    decision_ax.text(move.acceptance_probability, 0.75, "$\\alpha$", ha="left")
    decision_ax.legend(loc="upper right", fontsize=8)

axes[0, 0].set_xlabel("State $x$")
axes[0, 1].set_xlabel("State $x$")
plt.show()

For the first move, $\Delta\log p=-0.5$, so the acceptance probability is $e^{-0.5}\approx0.61$. Since $U=0.40$ is below that threshold, the chain moves to $x=1$. For the second move, $\Delta\log p=-4$, so the acceptance probability is only $e^{-4}\approx0.018$. The draw $U=0.10$ is too large, so the proposed value $x=3$ is rejected and the chain stays at $x=1$. The resulting chain states are $0,1,1$.

In the next activity, the proposal is generated from a Gaussian random walk and the target is Lecture 17’s `log_posterior`. The same four operations appear in the completed example: evaluate both log posteriors, subtract, compare $\log U$ with the acceptance limit, and either move or repeat the current state.

## Part 3: In-class coding activity 1 - Adapt the worked transition to Ramsey data

Use the completed one-dimensional transition above as a template. Complete the four marked TODO operations in the sampler: draw a vector proposal, evaluate the supplied Ramsey log posterior, calculate the log-posterior change, and apply the same log-space acceptance decision. The current state and proposal scales are ordered as $(\tau,C)$. A **trace plot** shows a parameter value against the sampler step. Inspect these traces and mark where warmup ends. After the baseline run, use the random-walk settings table to plan one-at-a-time comparisons. In Part 4, compare the retained posterior distributions and summaries. Lecture 19 develops formal diagnostics for judging whether finite chains have explored the target adequately.

In [ ]:
# Generate a random-walk Metropolis-Hastings chain for a supplied log posterior.
# The function returns every state and the fraction of proposals that were accepted.
def metropolis_hastings(log_posterior_function, initial_state, proposal_scales,
                        n_steps, rng):
    # Copy the initial parameter vector so later proposals do not alter the input.
    current = np.asarray(initial_state, dtype=float).copy()
    current_log_probability = log_posterior_function(current)
    # Reserve one row per step and one column per fitted parameter.
    chain = np.empty((n_steps, current.size))
    accepted = 0

    for step in range(n_steps):
        # TODO: Draw a Gaussian random-walk proposal around the current state.
        proposal = ...
        # TODO: Evaluate the proposed log posterior.
        proposal_log_probability = ...
        # TODO: Calculate proposed minus current log probability.
        log_probability_change = ...

        # TODO: Accept using log(U) < min(0, log_probability_change).
        accept_proposal = ...
        if accept_proposal:
            current = proposal
            current_log_probability = proposal_log_probability
            accepted += 1

        # Record every state, including repetitions after rejection, to preserve dwell time.
        chain[step] = current

    return chain, accepted / n_steps

In [ ]:
# Run the completed sampler from a stated starting point with reproducible randomness.
chain_rng = np.random.default_rng(1801)
chain, acceptance_fraction = metropolis_hastings(
    log_posterior,
    initial_state=np.array([1.0, 0.45]),
    proposal_scales=np.array([0.25, 0.040]),
    n_steps=18000,
    rng=chain_rng,
)
# Label the two parameter columns and add a step index for trace plotting.
chain_df = pd.DataFrame(chain, columns=["tau", "contrast"])
chain_df["step"] = np.arange(len(chain_df))

# Fixed classroom cutoff for this seeded example; warmup is not inferred automatically.
warmup_steps = 3000
print(f"Acceptance fraction = {acceptance_fraction:.3f}")

# Plot each parameter against step so the startup transient and warmup cutoff are visible.
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
axes[0].plot(chain_df["step"], chain_df["tau"], lw=0.5)
axes[0].axhline(
    true_parameters["tau"], color="black", ls="--",
    label=f"Generating $\\tau$ = {true_parameters['tau']:.2f} $\\mu$s",
)
axes[0].axvline(warmup_steps, color="C3", ls=":", label="Warmup ends")
axes[0].set(ylabel=r"$\tau$ ($\mu$s)", title="Trace")
axes[0].legend(fontsize=8)
axes[1].plot(chain_df["step"], chain_df["contrast"], lw=0.5)
axes[1].axhline(
    true_parameters["contrast"], color="black", ls="--",
    label=f"Generating contrast = {true_parameters['contrast']:.2f}",
)
axes[1].axvline(warmup_steps, color="C3", ls=":", label="Warmup ends")
axes[1].set(xlabel="Step", ylabel="Contrast")
axes[1].legend(fontsize=8)
plt.show()

### Warmup: moving from initialization into posterior exploration

A **stationary distribution** is a distribution that remains unchanged after another sampler step. The Metropolis-Hastings transition is constructed so that the Ramsey posterior is its stationary distribution. A chain started at an arbitrary point can take time to reach that regime; we call these initial steps **warmup** and exclude them from posterior summaries. The vertical line in the trace marks the chosen warmup cutoff. A stable-looking trace is useful evidence, but it does not by itself prove that the chain has reached stationarity.

This teaching sampler keeps its proposal scales fixed, so its warmup removes the transient from the chosen starting point. In more advanced samplers, warmup can also be used to tune or adapt proposal settings. The **acceptance fraction** is the number of proposed moves accepted divided by the total number of attempted moves. Proposal tuning changes how efficiently the chain explores the same posterior, not the posterior itself. For this two-dimensional random walk, an acceptance fraction around 0.2–0.5 is a useful heuristic, not a convergence proof.

The dashed generating values are references, not values the chain should settle on. MCMC explores the posterior supported by this finite data set; its center can differ from the generating values because of statistical fluctuations. Use the trace and sensitivity to the warmup cutoff as preliminary checks, while remembering that a stable-looking trace alone does not establish convergence. Lecture 19 develops stronger diagnostics.

There is no universal equation that determines the number of warmup steps. In practice, inspect the trace and compare posterior summaries after moving the cutoff earlier or later. For this seeded example, we discard the first 3,000 steps. This cutoff is illustrative, not a value calculated by the sampler or a general rule. Because proposal scales are fixed here, warmup only removes the initial transient. Lecture 19 returns to this choice when comparing multiple chains.

### Random-walk settings to explore

Change one sampler setting at a time and compare the retained posterior summaries from Part 4. With enough exploration, all settings that use the same likelihood, prior, and data should approximate the **same posterior**. Different paths are expected; materially different posterior summaries suggest that one or more runs are too short, poorly tuned, or stuck in different regions.

| Setting | What to try | What it changes | How to judge the choice |
|---|---|---|---|
| `initial_state` | Start at several different points inside the prior bounds, including dispersed points. | Where the chain begins; not the posterior target. | After warmup, traces and posterior summaries should agree across starts. A start outside the bounds has log posterior $-\infty$ and is invalid. |
| `proposal_scales` | Try smaller and larger step sizes for $\tau$ and $C$ separately. | How far each random-walk proposal moves. | Very small steps are often accepted but explore slowly; very large steps are often rejected. Compare trace movement and acceptance fraction; Lecture 19 develops additional sampling diagnostics. |
| `n_steps` | Run a shorter and a longer chain with the same target and seed. | How many transitions are available to map the posterior. | More steps generally improve the numerical precision of posterior summaries once sampling is adequate; they should not systematically change the posterior width or center. |
| `warmup_steps` | Compare reasonable cutoffs, for example 1,500, 3,000, and 4,500 for this run. | How many initial states are excluded from summaries. | Use the trace and compare posterior means and intervals across cutoffs. No cutoff is correct by formula; Lecture 19 develops additional checks using multiple chains. |
| `seed` | Repeat the same settings with different sampler seeds. | The particular random proposals and resulting path. | Independent runs should give compatible posterior summaries within their Monte Carlo error. Do not choose a seed because it puts the answer closest to the generating values. |

For this notebook, proposal scales are the available proposal-shape controls; the proposal covariance is diagonal. The acceptance rule itself is not a tuning parameter. Changing the prior bounds, likelihood, fixed $\omega$ or $\phi$, or observed data changes the inference problem rather than tuning the sampler.

## Part 4: From retained MCMC states to a posterior distribution

After warmup, each recorded state $\boldsymbol{\psi}^{(s)}=(\tau^{(s)},C^{(s)})$ is a draw from the posterior $p(\boldsymbol{\psi}\mid D_{\mathrm{obs}})$, approximately. The chain does not calculate a formula for the posterior at every point. Instead, it spends more steps in regions with higher posterior density. The empirical distribution of the retained states therefore approximates the posterior.

Keep every state after warmup, including repeated states caused by rejected proposals. Removing repeats would discard the chain's dwell time and distort the posterior. A histogram of $\tau$ estimates its marginal posterior; a histogram of $C$ does the same for contrast. The cloud of paired $(\tau,C)$ draws shows their joint posterior and any correlation. Posterior means and medians are averages and middle values of the retained samples. A central 68% interval runs from the 16th to the 84th percentile; a central 95% interval runs from the 2.5th to the 97.5th percentile. Here, we call the 16th–84th percentile range of posterior samples a 68% confidence interval, following the terminology used in this course. The covariance matrix summarizes how the two parameters vary together: positive covariance means larger sampled values of one tend to accompany larger values of the other, while negative covariance means one tends to increase as the other decreases. The plotted correlation coefficient $r$ is covariance divided by the product of the two standard deviations, so it is scale-free and lies between $-1$ and $1$.

These summaries are the scientific results we want from MCMC. Posterior width describes uncertainty in the parameters. Because the samples are correlated, the finite chain's numerical precision also depends on how efficiently it explores the posterior; Lecture 19 develops diagnostics for that question.

In [ ]:
# Keep all post-warmup states, including repeats following rejected proposals.
posterior_draws_df = chain_df.iloc[warmup_steps:].reset_index(drop=True)
parameter_names = ["tau", "contrast"]

# Summarize each marginal posterior with its mean, median, and central intervals.
summary_rows = []
for parameter_name in parameter_names:
    posterior_values = posterior_draws_df[parameter_name]
    summary_rows.append({
        "parameter": parameter_name,
        "posterior_mean": posterior_values.mean(),
        "posterior_median": posterior_values.median(),
        "68% confidence interval lower": posterior_values.quantile(0.16),
        "68% confidence interval upper": posterior_values.quantile(0.84),
        "95% confidence interval lower": posterior_values.quantile(0.025),
        "95% confidence interval upper": posterior_values.quantile(0.975),
    })

posterior_summary_df = pd.DataFrame(summary_rows)
posterior_covariance_df = posterior_draws_df[parameter_names].cov()
display(posterior_summary_df)
display(posterior_covariance_df)

In [ ]:
# Plot the two marginal posterior distributions and the joint sample cloud.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

# The first two panels show one-parameter marginal distributions.
axes[0].hist(posterior_draws_df["tau"], bins=45, density=True, alpha=0.75)
axes[0].axvline(
    true_parameters["tau"], color="black", ls="--",
    label=f"Generating $\\tau$ = {true_parameters['tau']:.2f} $\\mu$s",
)
axes[0].set(xlabel=r"$\tau$ ($\mu$s)", ylabel="Posterior density",
            title=r"Marginal posterior for $\tau$")
axes[0].legend(fontsize=8)

axes[1].hist(posterior_draws_df["contrast"], bins=45, density=True, alpha=0.75)
axes[1].axvline(
    true_parameters["contrast"], color="black", ls="--",
    label=f"Generating contrast = {true_parameters['contrast']:.2f}",
)
axes[1].set(xlabel="Contrast $C$", ylabel="Posterior density",
            title="Marginal posterior for contrast")
axes[1].legend(fontsize=8)

# The final panel shows the joint sample cloud and its linear correlation.
posterior_correlation = posterior_draws_df["tau"].corr(posterior_draws_df["contrast"])
axes[2].scatter(posterior_draws_df["tau"], posterior_draws_df["contrast"],
                s=3, alpha=0.12, label="Retained chain states")
axes[2].scatter(
    [true_parameters["tau"]], [true_parameters["contrast"]],
    marker="*", s=150, color="gold", edgecolor="black", zorder=4,
    label="Generating values",
)
axes[2].set(xlabel=r"$\tau$ ($\mu$s)", ylabel="Contrast $C$",
            title=f"Joint posterior samples (r = {posterior_correlation:.2f})")
axes[2].legend(fontsize=8)
plt.show()

### Production implementation: repeat the Ramsey analysis with `emcee`

The homegrown sampler made the transition rule visible. A production library handles the bookkeeping and offers tested sampling moves while using the same `log_posterior` from Lecture 17. Here `emcee` uses an **ensemble**: many walkers update using information from the other walkers. Its default affine-invariant stretch move is different from our independent Gaussian random-walk proposals, so the paths differ even though the target posterior is the same. We initialize the walkers in the plausible posterior region found above, discard a short burn-in, and then compare the production samples with the homegrown posterior.

`emcee` is included in the course `requirements.txt` (version 3.1.6). If you create a fresh environment, install the listed requirements before running this section.

In [ ]:
# Two parameters need an ensemble of walkers; 32 gives a broad starting ensemble.
emcee_parameter_names = ["tau", "contrast"]
emcee_ndim = len(emcee_parameter_names)
emcee_nwalkers = 32

# Center the initial ensemble on the homegrown posterior, then spread walkers
# by about one posterior standard deviation in each parameter.
walker_rng = np.random.default_rng(1819)
walker_center = posterior_draws_df[emcee_parameter_names].mean().to_numpy()
walker_spread = posterior_draws_df[emcee_parameter_names].std().to_numpy()
emcee_initial_positions = (
    walker_center
    + walker_rng.normal(size=(emcee_nwalkers, emcee_ndim)) * walker_spread
)

# Every starting walker must have a finite log posterior inside the prior support.
initial_log_posteriors = np.array([
    log_posterior(position) for position in emcee_initial_positions
])
if not np.all(np.isfinite(initial_log_posteriors)):
    raise ValueError("A starting walker is outside the valid prior region.")

The callback passed to `EnsembleSampler` is exactly the posterior function already used by the homemade chain. First let the walkers move away from their initial cloud. Then save their final positions, clear the burn-in records, and collect a fresh production run.

In [ ]:
# Seed emcee's random-number stream for a reproducible teaching run.
np.random.seed(1820)
emcee_sampler = emcee.EnsembleSampler(
    emcee_nwalkers, emcee_ndim, log_posterior
)

emcee_burnin_steps = 1000
emcee_production_steps = 5000

# Run burn-in and save the final walker positions for the production phase.
emcee_burnin_state = emcee_sampler.run_mcmc(
    emcee_initial_positions, emcee_burnin_steps
)

# Clear only the stored burn-in samples; keep the final walker state.
emcee_sampler.reset()
emcee_sampler.run_mcmc(
    emcee_burnin_state, emcee_production_steps
)

print("Production chain shape (steps, walkers, parameters):",
      emcee_sampler.get_chain().shape)

The production chain has shape `(steps, walkers, parameters)`. Each step stores the current parameter vector for every walker. Flattening the first two axes makes a table for posterior summaries and plots, but it does not make the walkers independent chains: each walker’s move depends on the ensemble. We compare the posterior distributions here; Lecture 19 covers sampler diagnostics.

In [ ]:
# Convert emcee's retained production draws into a labeled table.
emcee_flat_samples = emcee_sampler.get_chain(flat=True)
emcee_posterior_df = pd.DataFrame(
    emcee_flat_samples, columns=emcee_parameter_names
)

# Summarize the emcee posterior using the same quantiles as the homegrown chain.
emcee_summary_rows = []
for parameter_name in emcee_parameter_names:
    values = emcee_posterior_df[parameter_name]
    emcee_summary_rows.append({
        "parameter": parameter_name,
        "posterior_mean": values.mean(),
        "posterior_median": values.median(),
        "68% confidence interval lower": values.quantile(0.16),
        "68% confidence interval upper": values.quantile(0.84),
        "95% confidence interval lower": values.quantile(0.025),
        "95% confidence interval upper": values.quantile(0.975),
    })
emcee_summary_df = pd.DataFrame(emcee_summary_rows)
# Estimate the posterior covariance from the sampled parameter pairs.
emcee_covariance_df = emcee_posterior_df[emcee_parameter_names].cov()

# Compare central summaries and the estimated parameter correlation.
homegrown_correlation = posterior_draws_df["tau"].corr(
    posterior_draws_df["contrast"]
)
emcee_correlation = emcee_posterior_df["tau"].corr(
    emcee_posterior_df["contrast"]
)
summary_columns = [
    "parameter", "posterior_mean",
    "68% confidence interval lower", "68% confidence interval upper",
]
comparison_df = posterior_summary_df[summary_columns].merge(
    emcee_summary_df[summary_columns],
    on="parameter", suffixes=("_homegrown", "_emcee"),
)

# Summarize each walker's accepted-move fraction as sampler feedback.
walker_acceptance_df = pd.DataFrame({
    "walker": np.arange(emcee_nwalkers),
    "acceptance_fraction": emcee_sampler.acceptance_fraction,
})
print(f"Mean emcee walker acceptance fraction: "
      f"{walker_acceptance_df['acceptance_fraction'].mean():.3f}")
display(comparison_df)
print(f"Homegrown posterior correlation: {homegrown_correlation:.3f}")
print(f"emcee posterior correlation: {emcee_correlation:.3f}")
display(emcee_covariance_df)
display(walker_acceptance_df.head())

Compare the posterior means, intervals, covariance/correlation, and distributions, not the individual sample paths. Small differences are expected from Monte Carlo variation. The two samplers should tell a consistent scientific story for this dataset; if their summaries differ substantially, check initialization, chain length, and warmup. Lecture 19 develops formal sampling diagnostics.

In [ ]:
# Overlay the two samplers' marginal and joint posterior samples.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

# Compare the marginal distributions from both algorithms parameter by parameter.
axes[0].hist(posterior_draws_df["tau"], bins=45, density=True,
             histtype="step", lw=2, label="Homegrown MH")
axes[0].hist(emcee_posterior_df["tau"], bins=45, density=True,
             histtype="step", lw=2, label="emcee")
axes[0].axvline(true_parameters["tau"], color="black", ls="--",
                label="Generating value")
axes[0].set(xlabel=r"$\tau$ ($\mu$s)", ylabel="Posterior density",
            title=r"Marginal posterior for $\tau$")
axes[0].legend(fontsize=8)

axes[1].hist(posterior_draws_df["contrast"], bins=45, density=True,
             histtype="step", lw=2, label="Homegrown MH")
axes[1].hist(emcee_posterior_df["contrast"], bins=45, density=True,
             histtype="step", lw=2, label="emcee")
axes[1].axvline(true_parameters["contrast"], color="black", ls="--",
                label="Generating value")
axes[1].set(xlabel="Contrast $C$", ylabel="Posterior density",
            title="Marginal posterior for contrast")
axes[1].legend(fontsize=8)

# Plot only every tenth emcee draw to keep the overlaid joint cloud readable.
# The joint panel checks whether both algorithms recover a similar correlation pattern.
emcee_plot_df = emcee_posterior_df.iloc[::10]
axes[2].scatter(posterior_draws_df["tau"], posterior_draws_df["contrast"],
                s=3, alpha=0.12, label="Homegrown MH")
axes[2].scatter(emcee_plot_df["tau"], emcee_plot_df["contrast"],
                s=3, alpha=0.12, label="emcee")
axes[2].scatter([true_parameters["tau"]], [true_parameters["contrast"]],
                marker="*", s=130, color="gold", edgecolor="black",
                zorder=4, label="Generating values")
axes[2].set(xlabel=r"$\tau$ ($\mu$s)", ylabel="Contrast $C$",
            title="Joint posterior samples")
axes[2].legend(fontsize=8)
plt.show()

### In-class coding activity 2 - How many `emcee` steps are enough?

A sampler output is a sequence of evolving estimates. Compare posterior summaries from the first 500, first 1,500, and all production steps. This is a quick visual and numerical check of how the reported means and 68% intervals change as more samples are included. The prefixes are intentionally cumulative, so they are not independent estimates; formal autocorrelation and convergence checks follow in Lecture 19.

Complete the TODOs to flatten each prefix into a parameter table and calculate the posterior mean and central 68% interval for each parameter. Then inspect the summary table and the completed comparison plot. Do the estimates appear stable by the end of the run?

In [ ]:
# Keep the sampler's step and walker axes until selecting each cumulative prefix.
chain_by_step = emcee_sampler.get_chain()
prefix_lengths = [500, 1500, emcee_production_steps]
prefix_summary_rows = []

for prefix_length in prefix_lengths:
    # Flatten walkers and steps only after selecting the requested prefix.
    prefix_array = chain_by_step[:prefix_length].reshape(-1, emcee_ndim)
    prefix_df = pd.DataFrame(prefix_array, columns=emcee_parameter_names)

    for parameter_name in emcee_parameter_names:
        parameter_values = prefix_df[parameter_name]
        # TODO: Calculate the posterior mean for this prefix.
        prefix_mean = ...
        # TODO: Calculate the 16th and 84th percentiles for a central 68% interval.
        lower_68 = ...
        upper_68 = ...
        # TODO: Append a dictionary using keys production_steps, parameter, posterior_mean, lower_68%, and upper_68%.
        ...

prefix_summary_df = pd.DataFrame(prefix_summary_rows)
prefix_summary_df

In [ ]:
# Plot how posterior means and 68% intervals change with the number of retained steps.
fig, axes = plt.subplots(1, len(emcee_parameter_names), figsize=(10, 4), constrained_layout=True)
for ax, parameter_name in zip(axes, emcee_parameter_names):
    parameter_rows = prefix_summary_df[
        prefix_summary_df["parameter"] == parameter_name
    ]
    central = parameter_rows["posterior_mean"].to_numpy()
    lower_error = central - parameter_rows["lower_68%"].to_numpy()
    upper_error = parameter_rows["upper_68%"].to_numpy() - central
    ax.errorbar(parameter_rows["production_steps"], central,
                yerr=[lower_error, upper_error], marker="o", capsize=3)
    ax.set(xlabel="Production steps included", ylabel=parameter_name,
           title=f"Prefix stability: {parameter_name}")
plt.show()

## Part 5: Worked extension - infer frequency and phase as well

So far the Ramsey frequency $\omega$ and phase $\phi$ were treated as known, leaving a two-parameter posterior for decay time $\tau$ and contrast $C$. We can instead infer all four parameters from the same event counts. This is a useful demonstration of how MCMC handles a higher-dimensional posterior and how parameter correlations appear in pair plots.

Frequency and phase can be correlated: a small frequency shift can be partly offset by a phase shift over a finite observation window. In addition, a broad frequency search can produce alias solutions. To keep this worked example focused on joint uncertainty, we use a physically plausible frequency range around the generating frequency. The synthetic generating values are known for teaching; in a real experiment, starting points would come from prior scientific knowledge or an exploratory fit, and the results would need more extensive convergence checks.

In [ ]:
# Set explicit prior bounds for the four inferred Ramsey parameters.
omega_prior_min = 2.0 * np.pi * 0.70
omega_prior_max = 2.0 * np.pi * 1.00
phase_prior_min = -np.pi
phase_prior_max = np.pi


# Evaluate the four-parameter posterior for tau, contrast, angular frequency, and phase.
def log_posterior_four_parameters(parameters):
    # Unpack the parameter vector in the order used by the emcee sample table.
    tau, contrast, omega, phase = parameters

    # Apply uniform prior bounds, returning zero probability outside the support.
    inside_prior = (
        tau_prior_min <= tau <= tau_prior_max
        and contrast_prior_min <= contrast <= contrast_prior_max
        and omega_prior_min <= omega <= omega_prior_max
        and phase_prior_min <= phase <= phase_prior_max
    )
    if not inside_prior:
        return -np.inf

    # Evaluate the Ramsey probability for every measured delay.
    model_probability = ramsey_probability(
        ramsey_data_df["time_us"].to_numpy(), contrast, tau, omega, phase
    )

    # Sum binomial log probabilities for the observed excited-state counts.
    return np.sum(binom.logpmf(
        ramsey_data_df["excited_counts"].to_numpy(),
        ramsey_data_df["shots"].to_numpy(),
        model_probability,
    ))

Use 48 walkers for four dimensions and initialize them in a small cloud around a plausible point. The synthetic values are displayed as a reference because this is a worked simulation; they are not used in the likelihood. We first run burn-in, reset the stored chain, and then retain production draws, following the same procedure as the two-parameter `emcee` example.

In [ ]:
# Define parameter labels and the plausible center for this synthetic-data example.
four_parameter_names = ["tau", "contrast", "omega", "phase"]
four_parameter_center = np.array([
    true_parameters["tau"],
    true_parameters["contrast"],
    true_parameters["omega"],
    true_parameters["phase"],
])
initial_scales = np.array([0.15, 0.04, 0.08, 0.15])
four_nwalkers = 48
four_ndim = len(four_parameter_names)

# Scatter walkers around the plausible point, then check every start is allowed.
four_rng = np.random.default_rng(1821)
four_initial_positions = (
    four_parameter_center
    + four_rng.normal(size=(four_nwalkers, four_ndim)) * initial_scales
)
if not np.all(np.isfinite([
    log_posterior_four_parameters(position)
    for position in four_initial_positions
])):
    raise ValueError("A four-parameter starting walker is outside the prior bounds.")

# Run a separate emcee ensemble for the four-dimensional posterior.
np.random.seed(1822)
four_sampler = emcee.EnsembleSampler(
    four_nwalkers, four_ndim, log_posterior_four_parameters
)
four_burnin_state = four_sampler.run_mcmc(four_initial_positions, 2000)
four_sampler.reset()
four_sampler.run_mcmc(four_burnin_state, 8000)

# Store the retained draws in a labeled DataFrame for summaries and plotting.
four_parameter_samples_df = pd.DataFrame(
    four_sampler.get_chain(flat=True), columns=four_parameter_names
)
print("Four-parameter production chain shape:", four_sampler.get_chain().shape)
print("Mean walker acceptance fraction:",
      four_sampler.acceptance_fraction.mean())

### Read the pair plot and parameter correlations

A pair plot places each parameter's marginal distribution on the diagonal and shows every two-parameter sample cloud off the diagonal. Tilted or elongated clouds reveal correlated uncertainty; curved or separated shapes can reveal structure that a single covariance number does not capture. The annotated Pearson correlation coefficient summarizes linear association for each pair. It is dimensionless, while the sample covariance matrix carries parameter units, as in the covariance discussions in Lectures 4 and 16.

The pair plot is therefore a visual check of the joint posterior, not just four separate error bars. In particular, examine the frequency-phase panel and compare it with the $\tau$-contrast panel. A high correlation means the data constrain a combination of parameters more precisely than either parameter separately.

In [ ]:
# Calculate sample correlations and covariance from the retained posterior draws.
four_parameter_correlation = four_parameter_samples_df.corr()
four_parameter_covariance = four_parameter_samples_df.cov()
display(four_parameter_covariance)

# Build a corner-style pair plot: marginals on the diagonal, samples below,
# and correlation coefficients above the diagonal.
n_parameters = len(four_parameter_names)
fig, axes = plt.subplots(
    n_parameters, n_parameters,
    figsize=(11, 10),
    constrained_layout=True,
)
plot_samples_df = four_parameter_samples_df.iloc[::20]
parameter_labels = {
    "tau": r"$\tau$ ($\mu$s)",
    "contrast": r"$C$",
    "omega": r"$\omega$ (rad/$\mu$s)",
    "phase": r"$\phi$ (rad)",
}
# Diagonal cells show marginals, lower cells paired draws, and upper cells correlations.
for row, row_name in enumerate(four_parameter_names):
    for column, column_name in enumerate(four_parameter_names):
        ax = axes[row, column]
        # Draw the marginal, sample cloud, or correlation for this matrix cell.
        if row == column:
            ax.hist(four_parameter_samples_df[row_name], bins=45,
                    density=True, color="C0", alpha=0.75)
            ax.axvline(true_parameters[row_name], color="black", ls="--", lw=1)
        elif row > column:
            ax.scatter(plot_samples_df[column_name], plot_samples_df[row_name],
                       s=3, alpha=0.12, rasterized=True)
            ax.scatter(true_parameters[column_name], true_parameters[row_name],
                       marker="*", s=70, color="gold", edgecolor="black", zorder=3)
        else:
            correlation_value = four_parameter_correlation.loc[row_name, column_name]
            ax.text(0.5, 0.5, f"r = {correlation_value:.2f}",
                    ha="center", va="center", fontsize=13,
                    transform=ax.transAxes)
            ax.set_axis_off()

        if row == n_parameters - 1:
            ax.set_xlabel(parameter_labels[column_name])
        else:
            ax.set_xticklabels([])
        if column == 0 and row != column:
            ax.set_ylabel(parameter_labels[row_name])
        elif column != 0:
            ax.set_yticklabels([])

fig.suptitle(r"Ramsey posterior with $\tau$, contrast, $\omega$, and $\phi$", y=1.02)
plt.show()

# Compare posterior means and central 68% intervals with the known simulation inputs.
four_parameter_summary_rows = []
for parameter_name in four_parameter_names:
    values = four_parameter_samples_df[parameter_name]
    four_parameter_summary_rows.append({
        "parameter": parameter_name,
        "posterior_mean": values.mean(),
        "68% lower": values.quantile(0.16),
        "68% upper": values.quantile(0.84),
        "generating_value": true_parameters[parameter_name],
    })
four_parameter_summary_df = pd.DataFrame(four_parameter_summary_rows)
display(four_parameter_summary_df)

## Conclusion

- We built a Metropolis-Hastings sampler and used its retained states to estimate the Ramsey posterior.
- We reproduced the two-parameter analysis with `emcee`, then extended the worked example to infer decay time, contrast, frequency, and phase together.
- The marginal distributions and pair plots show parameter uncertainty and correlations that one-parameter summaries cannot capture.
- Lecture 19 will examine the numerical reliability of posterior samples using autocorrelation, effective sample size, and comparisons across chains.

## References and further reading

- [Unit 4 course resources](../resources/unit-4-monte-carlo-inference.md)
- [Hogg and Foreman-Mackey, *Data Analysis Recipes: Using MCMC*](https://arxiv.org/abs/1710.06068)
- [`emcee` quickstart](https://emcee.readthedocs.io/en/stable/tutorials/quickstart/)